In [129]:
import numpy as np
from scipy import linalg

In [130]:
def compute_M(F, x):
    n = len(x)
    J = jacobian_matrix(F, x)
    M = np.zeros(n)
    for i in range(n):
        M[i] = abs(J[i, i]) 
        
        M[i] = max(M[i], 1.0) * 125
    
    return M


def jacobian_matrix(F, x, h=1e-8):
    n = len(x)
    J = np.zeros((n, n))
    
    for i in range(n):
        for j in range(n):
            x_plus = x.copy()
            x_plus[j] += h
            f_x = F[i](*x)
            f_x_plus = F[i](*x_plus)
            
            J[i, j] = (f_x_plus - f_x) / h
    
    return J

In [131]:
def jakobi_nonlinear(F, x0, eps, max_iter = 100):
    n = len(x0)
    x = np.array(x0, dtype=float)
    M = compute_M(F, x)
    
    xs = [x.copy()]
    
    for _ in range(max_iter):
        x_new = np.zeros(n)
        for i in range(n):
            x_new[i] = x[i] - F[i](*x) / M[i]
        
        xs.append(x_new.copy())
        
        if np.max(np.abs(x_new - x)) < eps:
            break
        
        x = x_new.copy()
    
    return xs[-1]

In [132]:
def zeidel_nonlinear(F, x0, eps, max_iter=100):
    n = len(x0)
    x = np.array(x0, dtype=float)
    M = compute_M(F, x)
    
    xs = [x.copy()]
    
    for _ in range(max_iter):
        x_old = x.copy()
        for i in range(n):
            x[i] = x[i] - F[i](*x) / M[i]
        
        xs.append(x.copy())
        
        if np.max(np.abs(x - x_old)) < eps:
            break
        
    
    return xs[-1]

In [133]:
def newton_nonlinear(F, x0, eps, max_iter = 100):
    n = len(x0)
    x = np.array(x0, dtype=float)
    
    xs = [x.copy()]
    
    for _ in range(max_iter):
        J = jacobian_matrix(F, x)
        F_val = np.array([F[i](*x) for i in range(n)])
        try:
            delta = linalg.solve(J, -F_val)
        except linalg.LinAlgError:
            print("Матрица Якоби вырождена!")
            break

        x = x + delta
        xs.append(x.copy())

        if np.max(np.abs(delta)) < eps:
            break
    
    return xs[-1]

In [134]:
def F1(x, y):
    return np.sin(x + 1) + y - 1.3
    
def F2(x, y):
    return -np.sin(y - 1) + x - 0.8
    
x0 = [-0.6, -0.6]
eps = 0.001

In [135]:
solution = jakobi_nonlinear([F1, F2], x0, eps)

print(f"Решение: x = {solution[0]:.6f}, y = {solution[1]:.6f}")

Решение: x = 0.301396, y = -0.662974


In [136]:
solution = zeidel_nonlinear([F1, F2], x0, eps)

print(f"Решение: x = {solution[0]:.6f}, y = {solution[1]:.6f}")

Решение: x = 0.304280, y = -0.670941


In [137]:
solution = newton_nonlinear([F1, F2], x0, eps)
    
print(f"Решение: x = {solution[0]:.6f}, y = {solution[1]:.6f}")

Решение: x = 0.207142, y = 0.365397
